# Understand duplicate and unmatched cassette outputs

**CPU only: run all four code cells.** This notebook replays completed
predictions from the historical method and both arms of PR157 and
PR158. No photographs, model weights, GPU, training or fresh neural
inference are needed. Inputs total approximately 10.5 MB.

Three separate fixed probes remove completed outputs: require an
existing mutual reflected-view box association, or add suppression
above box overlap 0.60 or 0.50. Retained boxes, corner coordinates and
scores stay unchanged. Suppression is applied after the existing
inference pipeline; it does not replace native NMS. The settings are
illustrations of the duplicate-versus-missed-front tradeoff. None is
selected or promoted.

All three annotation references are reported. Unmatched outputs are
evaluation outcomes, not automatically physical negatives. Complete-pair
success is accompanied by detection of zero-, one- and two-known-corner
fronts. The same 20 photographs remain development data.


## 1 Load the fixed implementation

Source is pinned to an immutable commit. A changed source checkout is
preserved and rejected; choose a new `BAOMAI_DUPLICATE_AUDIT_BASE` if
rerunning beside edited files.


In [ ]:
from pathlib import Path
import hashlib, json, os, subprocess, sys

SOURCE_COMMIT = "9d457b56ee43a3514bd17a0e9070a5a243d69775"
SOURCE_HASHES = {'analyze_duplicate_audit_r66.py': '868ab2c5df33ba376da4212aca6281d99a9d5c3a9d6cd1dac7dfaf4e1cd7acfa', 'evaluate_pose.py': '3f80bd09efbe97327b705ffe6ad3e662111f3e2bc71b69b51a9d38d7baa90ce0', 'pose_scoring.py': '04e5cdbef26df978707d0b6e4b82f05471b6012845f7986b9c05c71644ec0018', 'pose_checkpoint.py': 'c1311307fd49b37e60f2939c9b084bc36690c1d13b6b31b9e4c51ea9353562d4', 'audit_reflection_return.py': '55f391535b040a27682e899aec74915b87864063023bc1ed818f8f82e67101de', 'predict_pose_folder.py': '2942864e86303a67911826fc0dbfe547b4b4fc2ed3860e74c9d0210c635520a6', 'run_corner_average_r58.py': '286232cf490f950b367068ea52b43078970ab27bb016169cb99159dab2fdbd88', 'run_original_comparison_r59.py': '279e2c0bd701299ea2a6f30403e97da68cc44a9c4ff874c5ee5a0d155a6e6ea3', 'run_reference_matrix_r60.py': '2939b2789abf2566d7680de60a0a87efae2a41e45c2dc01c9d15d1221cb8bbf1', 'run_training_label_comparison_r61.py': '0dc8d5dfdd344f590f88d17ea7ced76f3fc018026fae221a4df6e25d7c807a9c', 'run_flip_inference_r56.py': 'bf2ff9056f12976ca8c4aa971de2d2a7af1c679369c44dbf73de9f9d2542700d', 'diagnose_fronts_r50.py': 'cba4f1df6dcf431c933af1506a4de15718636d9477406eb34779ac4d8ab6ee3e', 'run_inference_ablation_r53.py': '8c8a37c3dcaccde5c568e9df6f44651cdb05965ca5f5ed5b0f2978852387dcc1'}
BASE = Path(os.environ.get("BAOMAI_DUPLICATE_AUDIT_BASE", "/content/baomai-duplicate-audit-r66")).resolve()
SOURCE, WORK = BASE / "source", BASE / "work"
INPUT_DIR, OUTPUT = WORK / "inputs", WORK / "results"
BASE.mkdir(parents=True, exist_ok=True)
ENV = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1", "GIT_TERMINAL_PROMPT": "0", "PYTHONDONTWRITEBYTECODE": "1"}
def run(command, cwd=None):
    subprocess.run(list(map(str, command)), cwd=cwd, env=ENV, check=True)
def git(*args):
    return subprocess.check_output(["git", *args], cwd=SOURCE, env=ENV, text=True).strip()
if not (SOURCE / ".git").exists():
    if SOURCE.exists() and any(SOURCE.iterdir()):
        raise RuntimeError("Existing files preserved; choose a fresh BAOMAI_DUPLICATE_AUDIT_BASE.")
    run(["git", "init", SOURCE])
    run(["git", "remote", "add", "origin", "https://github.com/Siuuuers/dl.git"], SOURCE)
    run(["git", "fetch", "--depth=1", "--filter=blob:none", "origin", SOURCE_COMMIT], SOURCE)
    run(["git", "sparse-checkout", "init", "--no-cone"], SOURCE)
    run(["git", "sparse-checkout", "set", "/scripts/", "/docs/thesis/evidence/annotation-reference/baomai-annotation-reference-r60-results.zip"], SOURCE)
    run(["git", "checkout", "--detach", SOURCE_COMMIT], SOURCE)
def verify_source():
    if git("rev-parse", "HEAD") != SOURCE_COMMIT or git("status", "--porcelain", "--untracked-files=all"):
        raise RuntimeError("Source changed; choose a fresh BAOMAI_DUPLICATE_AUDIT_BASE.")
    for name, expected in SOURCE_HASHES.items():
        if hashlib.sha256((SOURCE / "scripts" / name).read_bytes()).hexdigest() != expected:
            raise RuntimeError("Implementation hash differs: " + name)
verify_source()
WORK.mkdir(parents=True, exist_ok=True)
sys.dont_write_bytecode = True
sys.path.insert(0, str(SOURCE / "scripts"))
import analyze_duplicate_audit_r66 as audit
from IPython.display import Markdown, display
print("Immutable source:", SOURCE_COMMIT)


## 2 Verify the three saved-input archives

The historical reference archive is read from the pinned checkout.
PR157 and PR158 archives are fetched at their exact return commits if
not already present. Archive size, SHA-256, membership, CRC and every
inventoried member are checked. A corrupt existing input is rejected.
No checkpoint is downloaded or loaded.


In [ ]:
audit.stage_inputs(SOURCE, INPUT_DIR)
for name, spec in audit.INPUTS.items():
    print(name, spec["bytes"], "bytes; SHA-256", spec["sha256"])
print("Three immutable prediction archives verified.")


## 3 Replay the baselines and three fixed probes

Archived predictions, view associations, scores and cases must replay
exactly before the probes are scored. Each probe starts independently
from its unchanged method; filters are not combined. Evaluation matches
are recomputed after removal. Cross-view associations and averaged
coordinates are not recomputed.

The maximum-cardinality bipartite matching calculation is a diagnostic
upper bound on how many reference matches the same output boxes permit.
It is not an alternate published score and cannot be used at inference.


In [ ]:
summary, records = audit.analyze(SOURCE, INPUT_DIR)
audit.write_outputs(OUTPUT, summary, records)
display(Markdown((OUTPUT / "results.md").read_text()))
print("All-reference matrix rows:", len(summary["matrix"]))
print("Archived per-image baseline scores replayed:", summary["provenance"]["replayed_image_scores"])
print("Exact paired changes and removals:", summary["record_counts"])


## 4 Verify the saved evidence and download

Recompute the analysis and compare all saved result bytes. The ZIP
contains the all-reference matrix, availability strata, changed target
identities, removed-output records, baseline errors, retained original
indices, matching bounds and source/input provenance.

No labels or application defaults are changed. Fewer duplicates with
more missed partial fronts is reported as a tradeoff. This completed
CPU analysis does not provide independent capture-session performance.


In [ ]:
verify_source()
repeated_summary, repeated_records = audit.analyze(SOURCE, INPUT_DIR)
if repeated_summary != summary or repeated_records != records:
    raise RuntimeError("Independent saved-input replay differs")
CHECK = WORK / "verification"
audit.write_outputs(CHECK, repeated_summary, repeated_records)
inventory = json.loads((OUTPUT / "inventory.json").read_bytes())
for name in [*inventory, "inventory.json"]:
    if (OUTPUT / name).read_bytes() != (CHECK / name).read_bytes():
        raise RuntimeError("Saved result bytes differ: " + name)
RESULT = audit.package_results(OUTPUT, SOURCE_COMMIT)
manifest = {"source_commit": SOURCE_COMMIT, "notebook_code_cells": 4,
            "saved_output_replay": "byte-identical", "fresh_inference": False,
            "training": False, "labels_modified": False, "automatic_promotion": False,
            "result": {"filename": RESULT.name, "bytes": RESULT.stat().st_size,
                       "sha256": hashlib.sha256(RESULT.read_bytes()).hexdigest()}}
(WORK / "package-manifest.json").write_text(json.dumps(manifest, indent=2) + "\n")
print(json.dumps(manifest, indent=2))
from IPython.display import FileLink
try:
    from google.colab import files
except ImportError:
    display(FileLink(str(RESULT)))
else:
    files.download(str(RESULT))
